# AI Academic Writing Survey: Complete Pre-Test Analysis
## Master's Students' Use of AI Tools in Academic Writing

**Survey Period:** September 2026

**Sample Size:** n = 23 master's students

This notebook provides a comprehensive analysis of the pre-test survey on AI tool usage, attitudes, and academic confidence among master's students at FH Münster.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
sns.set_theme(style="whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")

## 1. Load and Clean Data

In [ ]:
# Load the survey data
df = pd.read_csv('data/results-survey563129.csv')

print(f"Dataset shape: {df.shape}")
print(f"Respondents: {len(df)}")
print(f"\nFirst few columns: {df.columns[:5].tolist()}")
print(f"\nData loaded successfully.")

In [ ]:
# Function to clean column names
def clean_column_name(col):
    """Extract variable name from LimeSurvey column format"""
    if '. ' in str(col):
        return str(col).split('. ')[0].strip()
    return str(col).strip()

# Clean all column names
df.columns = [clean_column_name(c) for c in df.columns]

# Identify key column groups
tool_cols = [c for c in df.columns if c.startswith('BTools')]
use_cols = [c for c in df.columns if c.startswith('BUse')]
literacy_cols = [c for c in df.columns if c.startswith('BLiteracy')]
conf_cols = [c for c in df.columns if c.startswith('CConf')]
att_cols = [c for c in df.columns if c.startswith('EAtt')]
dgsp_cols = [c for c in df.columns if c.startswith('DGSP')]

print(f"Tool columns (n={len(tool_cols)}): {tool_cols[:3]}...")
print(f"Use case columns (n={len(use_cols)}): {use_cols[:3]}...")
print(f"Literacy columns (n={len(literacy_cols)}): {literacy_cols}")
print(f"Confidence columns (n={len(conf_cols)}): {conf_cols[:3]}...")
print(f"Attitude columns (n={len(att_cols)}): {att_cols}")
print(f"Scientific practice columns (n={len(dgsp_cols)}): {dgsp_cols}")

In [ ]:
# Convert yes/no columns to binary
for col in tool_cols + use_cols:
    if col in df.columns:
        df[col] = df[col].fillna('No').astype(str).str.strip().str.lower().eq('yes').astype(int)

# Convert Likert items to numeric
for col in literacy_cols + conf_cols + att_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data cleaning completed.")
print(f"\nSample of converted data:")
print(df[[tool_cols[0], use_cols[0], literacy_cols[0], conf_cols[0]]].head())

## 2. AI Use Frequency Analysis

In [ ]:
# Analyze AI use frequency
freq_col = 'BFreq'
freq_counts = df[freq_col].value_counts()

# Define the correct order
freq_order = ['Never', 'A few times', 'About monthly', 'Abut weekly', 'several times a week', 'About weekly', 'Daily']
freq_counts_ordered = df[freq_col].value_counts().reindex([f for f in freq_order if f in df[freq_col].unique()])

print("\n" + "="*60)
print("AI USE FREQUENCY (Past 6 months)")
print("="*60)
for freq, count in freq_counts_ordered.items():
    pct = (count / len(df)) * 100
    print(f"{freq:25s}: {count:2d} respondents ({pct:5.1f}%)")

# Create visualization
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if f in ['Never'] else '#ff7f0e' if f in ['A few times', 'About monthly'] else '#2ca02c' 
          for f in freq_counts_ordered.index]
bars = ax.bar(range(len(freq_counts_ordered)), freq_counts_ordered.values, color=colors, edgecolor='black', linewidth=1.2)
ax.set_xticks(range(len(freq_counts_ordered)))
ax.set_xticklabels(freq_counts_ordered.index, rotation=45, ha='right')
ax.set_ylabel('Number of Respondents', fontsize=12, fontweight='bold')
ax.set_xlabel('Frequency of AI Use', fontsize=12, fontweight='bold')
ax.set_title('Frequency of Generative AI Tool Use in Academic Work (Past 6 Months)', 
             fontsize=13, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, freq_counts_ordered.values)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, str(int(val)), 
            ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/01_ai_use_frequency.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Chart saved: outputs/01_ai_use_frequency.png")

## 3. AI Tools Usage Analysis

In [ ]:
# Extract clean tool names
tool_names = {
    'BTools[SQ1]': 'ChatGPT',
    'BTools[SQ2]': 'Claude',
    'BTools[SQ3]': 'Gemini',
    'BTools[SQ4]': 'Copilot',
    'BTools[SQ5]': 'Elicit',
    'BTools[SQ6]': 'NotebookLM',
    'BTools[SQ7]': 'Research Rabbit',
    'BTools[SQ8]': 'Consensus',
    'BTools[SQ9]': 'DeepL Write',
    'BTools[SQ10]': 'Perplexity',
    'BTools[SQ11]': 'None',
}

# Calculate tool usage
tool_usage = pd.DataFrame({
    'Tool': [tool_names.get(col, col) for col in tool_cols if col in tool_names],
    'Count': [df[col].sum() for col in tool_cols if col in tool_names]
}).sort_values('Count', ascending=False)

tool_usage['Percentage'] = (tool_usage['Count'] / len(df) * 100).round(1)

print("\n" + "="*70)
print("AI TOOLS USED FOR ACADEMIC WORK")
print("="*70)
for idx, row in tool_usage.iterrows():
    print(f"{row['Tool']:20s}: {int(row['Count']):2d} respondents ({row['Percentage']:5.1f}%)")

# Create visualization
fig, ax = plt.subplots(figsize=(12, 6))
colors_gradient = plt.cm.RdYlGn(np.linspace(0.3, 0.8, len(tool_usage)))
bars = ax.barh(tool_usage['Tool'], tool_usage['Count'], color=colors_gradient, edgecolor='black', linewidth=1.1)
ax.set_xlabel('Number of Respondents', fontsize=12, fontweight='bold')
ax.set_title('AI Tools Used for Academic Work (Multiple Selection)', fontsize=13, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

# Add percentage labels
for i, (bar, row) in enumerate(zip(bars, tool_usage.itertuples())):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, 
            f"{int(row.Count)} ({row.Percentage:.1f}%)", 
            va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('outputs/02_ai_tools_used.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Chart saved: outputs/02_ai_tools_used.png")

## 4. Academic Tasks Supported by AI

In [ ]:
# Extract clean task names
task_names = {
    'BUse[SQ1]': 'Understanding a concept',
    'BUse[SQ2]': 'Finding literature',
    'BUse[SQ3]': 'Summarising papers',
    'BUse[SQ4]': 'Structuring or outlining',
    'BUse[SQ5]': 'Writing text I submitted',
    'BUse[SQ6]': 'Rewriting my own text',
    'BUse[SQ7]': 'Translation',
    'BUse[SQ8]': 'Data analysis',
}

# Calculate task usage
task_usage = pd.DataFrame({
    'Task': [task_names.get(col, col) for col in use_cols if col in task_names],
    'Count': [df[col].sum() for col in use_cols if col in task_names]
}).sort_values('Count', ascending=False)

task_usage['Percentage'] = (task_usage['Count'] / len(df) * 100).round(1)

print("\n" + "="*70)
print("ACADEMIC TASKS SUPPORTED BY AI TOOLS")
print("="*70)
for idx, row in task_usage.iterrows():
    print(f"{row['Task']:30s}: {int(row['Count']):2d} respondents ({row['Percentage']:5.1f}%)")

# Create visualization
fig, ax = plt.subplots(figsize=(12, 6))
colors_task = plt.cm.Spectral(np.linspace(0.2, 0.8, len(task_usage)))
bars = ax.barh(task_usage['Task'], task_usage['Count'], color=colors_task, edgecolor='black', linewidth=1.1)
ax.set_xlabel('Number of Respondents', fontsize=12, fontweight='bold')
ax.set_title('Academic Tasks Supported by AI Tools (Multiple Selection)', fontsize=13, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

# Add percentage labels
for i, (bar, row) in enumerate(zip(bars, task_usage.itertuples())):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, 
            f"{int(row.Count)} ({row.Percentage:.1f}%)", 
            va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('outputs/03_academic_tasks.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Chart saved: outputs/03_academic_tasks.png")

## 5. AI Literacy and Awareness

In [ ]:
# AI Literacy items
literacy_labels = {
    'BLiteracy[SQ8]': 'Judge unreliable/invented\ninformation',
    'BLiteracy[SQ9]': 'Understand how tools\nproduce answers',
    'BLiteracy[SQ10]': 'Know university rules on\nAI use in written work',
    'BLiteracy[SQ11]': 'Clear on acceptable vs.\nunacceptable AI use',
}

# Calculate means
literacy_data = df[list(literacy_labels.keys())].apply(pd.to_numeric, errors='coerce')
literacy_means = literacy_data.mean().sort_values(ascending=False)

# Create readable labels
literacy_display = pd.DataFrame({
    'Item': [literacy_labels.get(k, k) for k in literacy_means.index],
    'Mean': literacy_means.values
})

print("\n" + "="*70)
print("AI LITERACY: MEAN AGREEMENT (Scale 1-5)")
print("="*70)
print(f"Overall mean: {literacy_data.mean().mean():.2f}\n")
for idx, row in literacy_display.iterrows():
    stars = '★' * int(row['Mean']) + '☆' * (5 - int(row['Mean']))
    print(f"{row['Item']:40s}: {row['Mean']:4.2f} {stars}")

# Create visualization
fig, ax = plt.subplots(figsize=(12, 6))
colors_lit = ['#2ca02c' if m >= 3.5 else '#ff7f0e' if m >= 2.5 else '#d62728' for m in literacy_display['Mean']]
bars = ax.barh(literacy_display['Item'], literacy_display['Mean'], color=colors_lit, edgecolor='black', linewidth=1.1)
ax.set_xlim(0, 5)
ax.axvline(3, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Neutral (3.0)')
ax.set_xlabel('Mean Agreement (1-5)', fontsize=12, fontweight='bold')
ax.set_title('AI Literacy: Student Understanding and Awareness', fontsize=13, fontweight='bold', pad=20)
ax.set_xticks([1, 2, 3, 4, 5])
ax.grid(axis='x', alpha=0.3)

# Add mean value labels
for bar, val in zip(bars, literacy_display['Mean']):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}', 
            va='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/04_ai_literacy.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Chart saved: outputs/04_ai_literacy.png")

## 6. Confidence in Research Tasks

In [ ]:
# Research confidence items
conf_labels = {
    'CConf[SQ12]': 'Narrow a broad topic\nto researchable question',
    'CConf[SQ13]': 'Identify research gap\nfrom existing literature',
    'CConf[SQ14]': 'Build systematic\nliterature search',
    'CConf[SQ15]': 'Judge quality &\ncredibility of sources',
    'CConf[SQ16]': 'Structure theoretical\nbackground',
    'CConf[SQ17]': 'Maintain clear\nargument line',
    'CConf[SQ18]': 'Choose & justify\nresearch method',
    'CConf[SQ19]': 'Interpret results\nagainst theory',
    'CConf[SQ20]': 'Identify & articulate\nlimitations',
    'CConf[SQ21]': 'Document sources\n& decisions',
}

# Calculate means
conf_data = df[list(conf_labels.keys())].apply(pd.to_numeric, errors='coerce')
conf_means = conf_data.mean().sort_values(ascending=False)

# Create readable dataframe
conf_display = pd.DataFrame({
    'Item': [conf_labels.get(k, k) for k in conf_means.index],
    'Mean': conf_means.values
})

print("\n" + "="*70)
print("RESEARCH CONFIDENCE: MEAN CONFIDENCE (Scale 1-5)")
print("="*70)
print(f"Overall mean: {conf_data.mean().mean():.2f}\n")
for idx, row in conf_display.iterrows():
    stars = '★' * int(row['Mean']) + '☆' * (5 - int(row['Mean']))
    print(f"{row['Item']:40s}: {row['Mean']:4.2f} {stars}")

# Create visualization
fig, ax = plt.subplots(figsize=(12, 8))
colors_conf = ['#2ca02c' if m >= 3.8 else '#ff7f0e' if m >= 3.2 else '#d62728' for m in conf_display['Mean']]
bars = ax.barh(conf_display['Item'], conf_display['Mean'], color=colors_conf, edgecolor='black', linewidth=1.1)
ax.set_xlim(0, 5)
ax.axvline(3, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Neutral (3.0)')
ax.set_xlabel('Mean Confidence (1-5)', fontsize=12, fontweight='bold')
ax.set_title('Research Confidence: Master\'s Students\' Self-Assessed Capability', fontsize=13, fontweight='bold', pad=20)
ax.set_xticks([1, 2, 3, 4, 5])
ax.grid(axis='x', alpha=0.3)

# Add mean value labels
for bar, val in zip(bars, conf_display['Mean']):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}', 
            va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('outputs/05_research_confidence.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Chart saved: outputs/05_research_confidence.png")

## 7. Attitudes Toward AI in Academic Work

In [ ]:
# Attitude items
att_labels = {
    'EAtt[SQ27]': 'Using AI feels like\na "grey area"',
    'EAtt[SQ28]': 'Comfortable telling\nsupervisor about AI use',
    'EAtt[SQ29]': 'Prefer being shown\nhow to solve vs. solution',
}

# Calculate means
att_data = df[list(att_labels.keys())].apply(pd.to_numeric, errors='coerce')
att_means = att_data.mean().sort_values(ascending=False)

# Create readable dataframe
att_display = pd.DataFrame({
    'Item': [att_labels.get(k, k) for k in att_means.index],
    'Mean': att_means.values
})

print("\n" + "="*70)
print("ATTITUDES TOWARD AI IN ACADEMIC WORK: MEAN AGREEMENT (Scale 1-5)")
print("="*70)
print(f"Overall mean: {att_data.mean().mean():.2f}\n")
for idx, row in att_display.iterrows():
    stars = '★' * int(row['Mean']) + '☆' * (5 - int(row['Mean']))
    print(f"{row['Item']:40s}: {row['Mean']:4.2f} {stars}")

# Create visualization
fig, ax = plt.subplots(figsize=(10, 6))
colors_att = ['#1f77b4', '#ff7f0e', '#2ca02c']
bars = ax.barh(att_display['Item'], att_display['Mean'], color=colors_att, edgecolor='black', linewidth=1.1)
ax.set_xlim(0, 5)
ax.axvline(3, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Neutral (3.0)')
ax.set_xlabel('Mean Agreement (1-5)', fontsize=12, fontweight='bold')
ax.set_title('Student Attitudes Toward AI in Academic Work', fontsize=13, fontweight='bold', pad=20)
ax.set_xticks([1, 2, 3, 4, 5])
ax.grid(axis='x', alpha=0.3)

# Add mean value labels
for bar, val in zip(bars, att_display['Mean']):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}', 
            va='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/06_attitudes.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Chart saved: outputs/06_attitudes.png")

## 8. Scientific Practice Knowledge

In [ ]:
# Scientific practice items
dgsp_items = {
    'DGSP1': 'Rephrase own work for fluency',
    'DGSP2': 'Verify AI-suggested references',
    'DGSP3': 'Authorship under DFG guidelines',
    'DGSP4': 'Handle contradictory findings',
    'DGSP5': 'Purpose of documenting research',
}

print("\n" + "="*70)
print("SCIENTIFIC PRACTICE: SCENARIO RESPONSES")
print("="*70)

for dgsp_col in dgsp_cols:
    item_name = dgsp_items.get(dgsp_col, dgsp_col)
    print(f"\n{item_name} ({dgsp_col}):")
    responses = df[dgsp_col].value_counts()
    for resp, count in responses.items():
        if pd.notna(resp):
            print(f"  {count:2d} ({count/len(df)*100:5.1f}%): {str(resp)[:70]}")

## 9. Correlation Analysis

In [ ]:
# Prepare data for correlation
corr_data = df[list(literacy_labels.keys()) + list(conf_labels.keys()) + list(att_labels.keys())].apply(pd.to_numeric, errors='coerce')
corr_matrix = corr_data.corr()

# Create correlation heatmap
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            xticklabels=False, yticklabels=False, ax=ax)

ax.set_title('Correlation Matrix: AI Literacy, Research Confidence, and Attitudes', 
             fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('outputs/07_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Chart saved: outputs/07_correlation_heatmap.png")

In [ ]:
# Find strongest correlations
print("\n" + "="*70)
print("STRONGEST CORRELATIONS (r > 0.60)")
print("="*70)

# Extract upper triangle of correlation matrix
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.60:
            corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_val))

corr_pairs = sorted(corr_pairs, key=lambda x: abs(x[2]), reverse=True)

if corr_pairs:
    for var1, var2, corr_val in corr_pairs[:10]:
        label1 = literacy_labels.get(var1) or conf_labels.get(var1) or att_labels.get(var1) or var1
        label2 = literacy_labels.get(var2) or conf_labels.get(var2) or att_labels.get(var2) or var2
        print(f"r = {corr_val:+.3f}: {label1[:35]:35s} ↔ {label2[:35]}")
else:
    print("No strong correlations (r > 0.60) found.")

## 10. Open-Ended Responses: Hardest Part of Academic Writing

In [ ]:
# Extract open-ended responses
hardest_responses = df['FHardest'].dropna()

print("\n" + "="*70)
print("OPEN-ENDED: WHAT IS THE HARDEST PART OF ACADEMIC WRITING?")
print("="*70)
print(f"\nTotal responses: {len(hardest_responses)}\n")

for i, resp in enumerate(hardest_responses.head(23), 1):
    if resp and str(resp).strip() not in ['.', '-', '']:
        print(f"{i:2d}. {resp}")

# Thematic analysis
themes = {
    'Argumentation/Logic': ['roter faden', 'argument', 'logic', 'flow', 'structure', 'thread'],
    'Scope/Topic Selection': ['scope', 'topic', 'research gap', 'finding', 'narrow'],
    'Literature Selection': ['literature', 'papers', 'relevant', 'sources', 'select', 'identify'],
    'Precision/Clarity': ['precise', 'clear', 'specific', 'formulate', 'impact'],
    'Writing Quality': ['writing', 'fluently', 'formulate', 'express'],
    'Methodology': ['methodology', 'method', 'hypothesis', 'analysis'],
}

theme_counts = {theme: 0 for theme in themes.keys()}
for resp in hardest_responses:
    resp_lower = str(resp).lower()
    for theme, keywords in themes.items():
        if any(kw in resp_lower for kw in keywords):
            theme_counts[theme] += 1
            break

theme_df = pd.DataFrame(list(theme_counts.items()), columns=['Theme', 'Count']).sort_values('Count', ascending=False)

print("\n" + "="*70)
print("THEMATIC ANALYSIS")
print("="*70)
for theme, count in theme_df.values:
    pct = count / len(hardest_responses) * 100
    print(f"{theme:25s}: {count:2d} mentions ({pct:5.1f}%)")

## 11. Open-Ended Responses: CLARA Expectations

In [ ]:
# Extract CLARA responses
clara_responses = df['FCLARAExpect'].dropna()

print("\n" + "="*70)
print("OPEN-ENDED: WHAT WOULD MAKE CLARA USEFUL? WHAT WOULD BE ANNOYING?")
print("="*70)
print(f"\nTotal responses: {len(clara_responses)}\n")

for i, resp in enumerate(clara_responses.head(23), 1):
    if resp and str(resp).strip() not in ['.', '-', '', 'Never used it. Maybe to get FH based results Incan rely on. ']:
        print(f"{i:2d}. {resp[:95]}..." if len(str(resp)) > 95 else f"{i:2d}. {resp}")

# Thematic analysis
clara_themes = {
    'Trusted/Source-Based Responses': ['source', 'literature', 'database', 'verified', 'traced', 'reliable', 'accurate', 'made up'],
    'Context Retention': ['context', 'remember', 'understanding', 'discussion', 'ongoing'],
    'Concise & Clear Answers': ['short', 'useful', 'not too long', 'concrete', 'direct'],
    'Institutional Alignment': ['guidelines', 'FH', 'rules', 'policies', 'plagiarism'],
    'Literature Support': ['literature', 'research', 'papers', 'summarize', 'help'],
    'Document Upload': ['upload', 'documents', 'assist'],
}

clara_theme_counts = {theme: 0 for theme in clara_themes.keys()}
for resp in clara_responses:
    resp_lower = str(resp).lower()
    for theme, keywords in clara_themes.items():
        if any(kw in resp_lower for kw in keywords):
            clara_theme_counts[theme] += 1

clara_theme_df = pd.DataFrame(list(clara_theme_counts.items()), columns=['Theme', 'Count']).sort_values('Count', ascending=False)
clara_theme_df = clara_theme_df[clara_theme_df['Count'] > 0]

print("\n" + "="*70)
print("THEMATIC ANALYSIS: DESIRED CLARA FEATURES")
print("="*70)
for theme, count in clara_theme_df.values:
    pct = count / len(clara_responses) * 100
    print(f"{theme:35s}: {count:2d} mentions ({pct:5.1f}%)")

## 12. Summary Statistics Table

In [ ]:
# Create comprehensive summary table
summary_stats = {
    'Metric': [
        'Total Respondents',
        'Survey Completion Rate',
        '',
        'Daily AI Use',
        'Several Times/Week',
        'About Weekly',
        'Occasional (Monthly or Less)',
        '',
        'ChatGPT Users',
        'NotebookLM Users',
        'Gemini Users',
        'Perplexity Users',
        '',
        'AI Literacy (Mean)',
        'Research Confidence (Mean)',
        'Attitudes Toward AI (Mean)',
        '',
        'Strongest Confidence Task',
        'Weakest Confidence Task',
        '',
        'Open-Ended Responses Received',
    ],
    'Value': [
        f"{len(df)}",
        "100%",
        '',
        f"{(df['BFreq']=='Daily').sum()} ({(df['BFreq']=='Daily').sum()/len(df)*100:.1f}%)",
        f"{(df['BFreq']=='several times a week').sum()} ({(df['BFreq']=='several times a week').sum()/len(df)*100:.1f}%)",
        f"{((df['BFreq']=='Abut weekly') | (df['BFreq']=='About weekly')).sum()} ({((df['BFreq']=='Abut weekly') | (df['BFreq']=='About weekly')).sum()/len(df)*100:.1f}%)",
        f"{((df['BFreq']=='A few times') | (df['BFreq']=='About monthly')).sum()} ({((df['BFreq']=='A few times') | (df['BFreq']=='About monthly')).sum()/len(df)*100:.1f}%)",
        '',
        f"{df['BTools[SQ1]'].sum()} ({df['BTools[SQ1]'].sum()/len(df)*100:.1f}%)",
        f"{df['BTools[SQ6]'].sum()} ({df['BTools[SQ6]'].sum()/len(df)*100:.1f}%)",
        f"{df['BTools[SQ3]'].sum()} ({df['BTools[SQ3]'].sum()/len(df)*100:.1f}%)",
        f"{df['BTools[SQ10]'].sum()} ({df['BTools[SQ10]'].sum()/len(df)*100:.1f}%)",
        '',
        f"{literacy_data.mean().mean():.2f} / 5.00",
        f"{conf_data.mean().mean():.2f} / 5.00",
        f"{att_data.mean().mean():.2f} / 5.00",
        '',
        conf_display.iloc[0]['Item'].replace('\n', ' '),
        conf_display.iloc[-1]['Item'].replace('\n', ' '),
        '',
        f"{len(hardest_responses)} (100%)",
    ]
}

summary_df = pd.DataFrame(summary_stats)

print("\n" + "="*80)
print("SUMMARY STATISTICS TABLE")
print("="*80)
print(summary_df.to_string(index=False))

# Save table to CSV
summary_df.to_csv('outputs/00_summary_table.csv', index=False)
print(f"\n✓ Summary table saved: outputs/00_summary_table.csv")

## 13. Create All Summary Outputs

In [ ]:
# Ensure output directory exists
os.makedirs('outputs', exist_ok=True)

# Save cleaned dataset
df.to_csv('outputs/cleaned_survey_data.csv', index=False)

print("\n" + "="*70)
print("DATA EXPORT COMPLETE")
print("="*70)
print(f"✓ Cleaned survey data: outputs/cleaned_survey_data.csv")
print(f"✓ Summary table: outputs/00_summary_table.csv")
print(f"✓ Charts generated: 07 visualizations")
print(f"\nAll outputs saved to the 'outputs/' directory.")

---

## Analysis Complete

This notebook has generated:

1. **Descriptive Statistics** — frequency, tool usage, task categories
2. **Likert Scale Analysis** — AI literacy, research confidence, attitudes
3. **Correlation Analysis** — relationships between variables
4. **Thematic Analysis** — open-ended responses coded into themes
5. **Visualizations** — publication-ready charts and heatmaps
6. **Summary Tables** — exportable results for thesis/report

**Next Steps:**
- Use these outputs for your thesis results chapter
- Repeat this notebook with post-test data for comparison
- Export charts directly into your manuscript
- Use summary statistics for inferential analysis or qualitative discussion